# 03 — HITL & Constitutional AI

**Session: Month 5, Session 2**

## Learning Objectives
1. Understand when and why to pause agent execution for human approval
2. Walk through the HITL approval lifecycle
3. Apply Constitutional AI principles to agent inputs and outputs
4. Implement a principle violation handler

## Human-in-the-Loop Patterns

```
Fully Autonomous ←————————————→ Fully Manual
     [Agent acts]    [HITL threshold]    [Agent suggests]
     Low risk              ↑              High risk
     < 0.5              0.5-0.7             > 0.7
```

The HITL threshold is a business decision, not a technical one.
For a bank: threshold = 0.3. For a pizza shop: threshold = 0.9.

In [ ]:
import sys
sys.path.insert(0, '../backend')

import asyncio
from app.guardrails.hitl import HITLManager, MemoryTransport
from app.guardrails.constitutional import ConstitutionalGuard

## Part 1: HITL Approval Workflow

In [ ]:
manager = HITLManager(transport=MemoryTransport())

async def demo_hitl_approve():
    print('=== HITL Demo: Approve Flow ===')
    
    # Step 1: Agent requests approval for a $750 refund
    req = await manager.request_approval(
        tool_name='process_refund',
        args={'order_id': 'ORD-10001', 'amount': 750.0, 'reason': 'Major defect'},
        reason='Refund exceeds $500 — supervisor approval required',
        risk_score=0.75,
        requested_by='conv-demo-001',
    )
    print(f'\n1. Approval request created: {req.approval_id}')
    print(f'   Tool: {req.tool_name} | Amount: ${req.args["amount"]}')
    print(f'   Status: {req.status.value}')
    
    # Step 2: Supervisor approves
    print('\n2. Supervisor approves the request...')
    manager.approve(req.approval_id, 'jane.supervisor@company.com')
    
    # Step 3: Agent waits and gets result
    approved = await manager.wait_for_approval(req.approval_id, poll_interval=0.05)
    print(f'3. Agent received: {"APPROVED ✅" if approved else "DENIED ❌"}')
    
    updated = manager.get_request(req.approval_id)
    print(f'   Resolved by: {updated.resolved_by}')
    print(f'   Resolved at: {updated.resolved_at}')

await demo_hitl_approve()

In [ ]:
async def demo_hitl_deny():
    print('\n=== HITL Demo: Deny Flow ===')
    
    req = await manager.request_approval(
        tool_name='cancel_order',
        args={'order_id': 'ORD-10002', 'reason': 'Suspicious request'},
        reason='Refund > $500',
        risk_score=0.80,
    )
    print(f'\n1. Approval request created: {req.approval_id}')
    
    # Supervisor denies
    manager.deny(req.approval_id, 'john.supervisor@company.com', reason='Customer identity not verified')
    print('2. Supervisor denied the request')
    
    approved = await manager.wait_for_approval(req.approval_id, poll_interval=0.05)
    print(f'3. Agent received: {"APPROVED" if approved else "DENIED ❌"}')
    
    updated = manager.get_request(req.approval_id)
    print(f'   Denial reason: {updated.denial_reason}')

await demo_hitl_deny()

## Part 2: Constitutional AI Checks

In [ ]:
guard = ConstitutionalGuard()

print('=== Constitutional Input Checks ===')
input_scenarios = [
    {
        'user': 'Please cancel my order',
        'tool': 'cancel_order',
        'args': {'order_id': 'ORD-1', 'reason': 'Customer request'},
    },
    {
        'user': 'I am just wondering about my delivery estimate',  # vague — no cancel intent
        'tool': 'cancel_order',
        'args': {'order_id': 'ORD-1', 'reason': 'Agent assumption'},
    },
    {
        'user': 'What is my order status?',  # narrow question
        'tool': 'get_customer_order_history',  # broad tool
        'args': {'customer_id': 'CUST-1'},
    },
    {
        'user': 'Maybe cancel my order, I am not sure',
        'tool': 'cancel_order',
        'args': {'order_id': 'ORD-1', 'reason': 'Not sure'},
    },
]

for s in input_scenarios:
    result = guard.check_input(s['user'], s['tool'], s['args'])
    status = '✅ PASSED' if result.passed else '⚠️  WARNING'
    print(f'\nUser:    {s["user"][:55]}')
    print(f'Tool:    {s["tool"]}')
    print(f'Result:  {status} (score={result.overall_score:.2f})')
    if result.warnings:
        for w in result.warnings:
            print(f'  ⚠️  [{w["principle"]}] {w["detail"]}')

In [ ]:
print('\n=== Constitutional Output Checks ===')
output_scenarios = [
    {
        'user': 'When will my refund arrive?',
        'response': 'Your refund has been processed and typically takes 3-5 business days to appear.',
    },
    {
        'user': 'When will my refund arrive?',
        'response': 'I guarantee your refund will definitely be in your account within 2 hours.',
    },
    {
        'user': 'Has my order shipped?',
        'response': 'Don\'t worry, I already expedited your shipment without you asking.',
    },
    {
        'user': 'What is my order status?',
        'response': 'Your order ORD-10001 is currently in processing status.',
    },
]

for s in output_scenarios:
    result = guard.check_output(s['user'], s['response'])
    status = '✅ PASSED' if result.passed else '🚫 VIOLATION'
    print(f'\nUser:     {s["user"]}')
    print(f'Response: {s["response"][:70]}')
    print(f'Result:   {status} (score={result.overall_score:.2f})')
    if result.violations:
        for v in result.violations:
            print(f'  🚫 [{v["principle"]}] {v["detail"]}')

## Lab Exercise: Define Your Own Constitution

Add 2 new principles to the `ConstitutionalGuard`:

**Principle 1: Empathy** 
Responses to customer complaints must acknowledge the customer's frustration  
before offering a solution.

**Principle 2: Accuracy**
The agent must not claim to have performed an action it hasn't taken yet.
(e.g., saying 'I have cancelled your order' before calling the tool)

For each principle:
1. Define the `Principle` dataclass
2. Implement the check in `_evaluate_output_principle()`
3. Write 2 test cases in `test_guardrails.py`